# Host Transcriptomic Response to M. tuberculosis Infection

In [ ]:
import GEOparse
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import zscore
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print("Loading dataset...")
gse = GEOparse.get_GEO(geo="GSE37250", destdir="./")
#----------------------------------------------------------------
sample_info = []
for gsm_id, gsm in gse.gsms.items():
    chars = gsm.metadata.get('characteristics_ch1', [])
    disease_state = hiv_status = ''
    for c in chars:
        if 'disease state:' in c:
            disease_state = c.replace('disease state:', '').strip()
        elif 'hiv status:' in c:
            hiv_status = c.replace('hiv status:', '').strip()
    sample_info.append({'gsm_id': gsm_id,
                        'disease_state': disease_state,
                        'hiv_status': hiv_status})

df_meta = pd.DataFrame(sample_info)
df_filtered = df_meta[
    (df_meta['hiv_status'] == 'HIV negative') &
    (df_meta['disease_state'].isin(['active tuberculosis',
                                    'latent TB infection']))
].copy()
print(f"Samples: {df_filtered['disease_state'].value_counts().to_dict()}")

In [ ]:
#-------------------------------------------------------------------
expression_data = {}
for gsm_id in df_filtered['gsm_id']:
    gsm = gse.gsms[gsm_id]
    expression_data[gsm_id] = gsm.table.set_index('ID_REF')['VALUE']

expr_matrix = pd.DataFrame(expression_data)
#--------------------------------------------------------------------
platform = gse.gpls[list(gse.gpls.keys())[0]]
annotation = platform.table[['ID', 'Symbol']].dropna(subset=['Symbol'])
annotation = annotation[annotation['Symbol'] != '']

expr_ann = expr_matrix.reset_index()
expr_ann = expr_ann.merge(annotation,
                          left_on='ID_REF', right_on='ID', how='inner')
expr_ann['mean_expr'] = expr_ann.iloc[:, 1:-2].mean(axis=1)
expr_ann = (expr_ann.sort_values('mean_expr', ascending=False)
                    .drop_duplicates(subset='Symbol', keep='first')
                    .set_index('Symbol')
                    .drop(columns=['ID_REF', 'ID', 'mean_expr']))
#---------------------------------------------------------------------
expr_clean = expr_ann[expr_ann > 0]
expr_clean = (expr_clean
              .dropna(thresh=int(0.5 * expr_clean.shape[1]))
              .fillna(0))
print(f"Genes after filtering: {expr_clean.shape[0]}")

#---------------------------------------------------------------------
labels = df_filtered.set_index('gsm_id')['disease_state']
labels = labels.loc[expr_clean.columns]

#---------------------------------------------------------------------
active_expr = expr_clean[labels[labels == 'active tuberculosis'].index]
latent_expr = expr_clean[labels[labels == 'latent TB infection'].index]
results = []
for gene in expr_clean.index:
    av = active_expr.loc[gene].values
    lv = latent_expr.loc[gene].values
    _, p_val = stats.ttest_ind(av, lv)
    ma, ml = av.mean(), lv.mean()
    log2fc = np.log2(ma / ml) if ma > 0 and ml > 0 else 0
    results.append({'gene': gene, 'log2fc': log2fc,
                    'p_value': p_val,
                    'mean_active': ma, 'mean_latent': ml})
df_results = pd.DataFrame(results)
_, p_adj, _, _ = multipletests(df_results['p_value'], method='fdr_bh')
df_results['p_adj'] = p_adj
df_results['significant'] = p_adj < 0.05
df_results = df_results.sort_values('p_adj')

up = ((df_results['p_adj'] < 0.05) & (df_results['log2fc'] > 0.5)).sum()
down = ((df_results['p_adj'] < 0.05) & (df_results['log2fc'] < -0.5)).sum()

print(f"\nSignificant DEGs (FDR<0.05): {df_results['significant'].sum()}")
print(f"Upregulated in active TB: {up}")
print(f"Downregulated in active TB: {down}")

#---------------------------------------------------------------------------
colors = []
for _, row in df_results.iterrows():
    if row['p_adj'] < 0.05 and row['log2fc'] > 0.5:
        colors.append('red')
    elif row['p_adj'] < 0.05 and row['log2fc'] < -0.5:
        colors.append('steelblue')
    else:
        colors.append('grey')
plt.figure(figsize=(10, 7))
plt.scatter(df_results['log2fc'],
            -np.log10(df_results['p_adj']),
            c=colors, alpha=0.4, s=8)

for _, row in df_results.head(15).iterrows():
    plt.annotate(row['gene'],
                 (row['log2fc'], -np.log10(row['p_adj'])),
                 fontsize=7, ha='center')

plt.axhline(-np.log10(0.05), color='black', linestyle='--', linewidth=0.8)
plt.axvline(0.5, color='black', linestyle=':', linewidth=0.8)
plt.axvline(-0.5, color='black', linestyle=':', linewidth=0.8)
plt.xlabel('Log2 Fold Change (Active TB vs Latent TB)', fontsize=12)
plt.ylabel('-log10(Adjusted P-value)', fontsize=12)
plt.title('Differential Gene Expression: Active TB vs Latent TB\n'
          '(HIV-negative whole blood, GSE37250)', fontsize=12)
from matplotlib.patches import Patch
plt.legend(handles=[
    Patch(facecolor='red', label='Upregulated in Active TB'),
    Patch(facecolor='steelblue', label='Downregulated in Active TB'),
    Patch(facecolor='grey', label='Not significant')
], fontsize=10)
plt.tight_layout()
plt.savefig('volcano_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#-----------------------------------------------------------------------------
top50 = df_results.head(50)['gene'].tolist()
heatmap_data = expr_clean.loc[top50]
heatmap_zscore = pd.DataFrame(
    zscore(heatmap_data.values, axis=1),
    index=heatmap_data.index,
    columns=heatmap_data.columns
)
sample_order = labels.sort_values().index.tolist()
heatmap_zscore = heatmap_zscore[sample_order]

plt.figure(figsize=(14, 10))
sns.heatmap(heatmap_zscore, cmap='RdBu_r', center=0,
            xticklabels=False, yticklabels=True,
            cbar_kws={'label': 'Z-score'})
plt.title('Top 50 Differentially Expressed Genes\n'
          'Active TB vs Latent TB (HIV-negative)', fontsize=13)
plt.xlabel('Samples', fontsize=11)
plt.ylabel('Genes', fontsize=11)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig('heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
df_results.to_csv('DE_results.csv', index=False)
#=============================================================================
print("\n" + "="*50)
print("FINAL SUMMARY")
print("="*50)
print(f"Dataset:     GSE37250")
print(f"Comparison:  Active TB vs Latent TB (HIV-negative)")
print(f"Samples:     97 active TB, 83 latent TB")
print(f"Genes tested: {len(df_results):,}")
print(f"Sig DEGs:    {df_results['significant'].sum():,}")
print(f"Upregulated: {up:,}")
print(f"Downregulated: {down:,}")
print(f"\nTop 5 upregulated:")
print(df_results[df_results['log2fc']>0][['gene','log2fc','p_adj']].head(5).to_string(index=False))
print(f"\nTop 5 downregulated:")
print(df_results[df_results['log2fc']<0][['gene','log2fc','p_adj']].head(5).to_string(index=False))